# Reasoning — Try it in PyTorch

An **optional** hands-on companion to [Chapter 11](https://learnai.robennals.org/reasoning). The chapter says a reasoning model writes its working into a message you never see, and that nothing about the architecture changes. Here you watch it happen, turn the working off, and see what it costs.

New to PyTorch? The [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) is a quick introduction.

## About the model used here

**Qwen3-1.7B** is small enough to run free in a few minutes, and hundreds of times smaller than the models behind ChatGPT or Claude. It makes mistakes they would not. What it does have is the same machinery, including a switch that turns its thinking on and off, which is exactly the comparison this chapter needs.

In [ ]:
!pip install -q transformers accelerate

import re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
print("running on", DEVICE)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", dtype=DTYPE).to(DEVICE)
print("model loaded")

## Everything the model emits

`enable_thinking` decides whether the prompt leaves room for the model to write to itself first. With it on, the model emits `<think>` tokens, works through the problem, closes the tag, and only then writes the answer.

Nothing here is a separate channel. It is one stream of tokens, and the app simply hides the part between the tags.

In [ ]:
def run(question, thinking=True, max_new_tokens=10_000, temperature=None):
    """Ask the model a question. Returns everything it emits, tags and all.

    The limit is set high on purpose. Thinking runs to a few hundred tokens for
    an easy question and well over a thousand for a hard one, and a model cut
    off mid-thought never writes an answer at all. It is not unlimited only
    because a small model can occasionally get stuck repeating itself, and a
    cap turns that into a wait rather than a hang.
    """
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=temperature is not None,
            temperature=temperature, top_p=0.95 if temperature else None,
            pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs.input_ids.shape[1]:]
    if len(new_tokens) >= max_new_tokens:
        print(f"! stopped at the {max_new_tokens} token limit rather than finishing. "
              "Raise max_new_tokens if you want to see the rest.")
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

BAT_AND_BALL = ("A bat and a ball cost $1.10 together. The bat costs $1.00 more "
                "than the ball. How much does the ball cost? Reply with just the amount.")

everything, n = run(BAT_AND_BALL)
print(f"{n} tokens emitted\n")
print(everything)

The `<think>` block is the model talking to itself. What the app would show you is only what comes after `</think>`.

In [ ]:
def split_thinking(text):
    """Separate the hidden working from the answer the user is shown."""
    if "</think>" in text:
        thinking, answer = text.split("</think>", 1)
        return thinking.replace("<think>", "").strip(), answer.strip()
    return "", text.strip()

thinking, answer = split_thinking(everything)
print("HIDDEN WORKING:", len(thinking.split()), "words")
print("WHAT YOU SEE  :", answer)

**Try your own.** Swap `BAT_AND_BALL` for a question of your own and look at the working rather than the answer. Things worth trying: a puzzle with a tempting wrong answer, a question needing several exact steps, and something with no right answer at all, where you can watch it think about a question that cannot be checked.

Watch the token count too. `max_new_tokens` is set high enough here that the model finishes on its own, but if you lower it and a question needs more, the model never closes its `<think>` tag and you get working with no answer on the end. That is not the model failing, that is us cutting it off.

## What the thinking costs

Every one of those thinking tokens takes time to generate, and on a paid model it costs money. The chapter warns that more thinking is not always better; this is the size of the bill for one easy question.

In [ ]:
import time

start = time.time()
text, n = run(BAT_AND_BALL, thinking=True)
seconds = time.time() - start

thinking, answer = split_thinking(text)
print(f"{n} tokens in {seconds:.0f} seconds")
print(f"{len(thinking.split())} words of working for a {len(answer.split())} word answer")

## How a reasoning model is trained, in miniature

The chapter says nobody marks the working. What gets marked is the final answer, and only for problems a computer can check.

The loop below is the part of that method you can run on a laptop. Ask the same question several times at a temperature above zero so the model takes a different path each time, then mark **only the final answer** with a few lines of Python. The attempts that got there are the ones real training would make more likely.

In [ ]:
LETTERS = ("How many times does the letter r appear in the word strawberry? "
           "Reply with just the number.")

def answer_is(text, expected):
    """Mark only the final answer, the way a checker would."""
    answer = split_thinking(text)[1]
    found = re.findall(r"\d+", answer)
    return bool(found) and found[0] == expected

# One attempt, taking the single most likely path every time.
greedy, n = run(LETTERS, thinking=True)
print(f"greedy attempt: {split_thinking(greedy)[1][:30]!r} "
      f"-> {'right' if answer_is(greedy, '3') else 'wrong'}")

Taking the most likely token at every step, this model walks the same wrong path every time. Now let the sampling wander, and mark each attempt with the checker.

In [ ]:
ATTEMPTS = 6
torch.manual_seed(0)

results = []
for attempt in range(ATTEMPTS):
    text, n = run(LETTERS, thinking=True, temperature=0.9)
    ok = answer_is(text, "3")
    results.append(ok)
    print(f"attempt {attempt + 1}: {'reached the answer' if ok else 'missed'}  "
          f"({n} tokens)")

print(f"\n{sum(results)} of {ATTEMPTS} attempts got there")
print("the checker can find the good ones without reading any of the working")

That is the whole trick. One deterministic attempt is stuck with whatever path it prefers, but a handful of sampled attempts explore different paths, and a checker can pick out the ones that arrived without anyone reading a single line of working.

Real training does exactly this at scale: sample many attempts at many problems, keep what reached the answer, and adjust the weights to make those more likely. Do it for long enough and the habits that show up in successful attempts — checking your own claim, finishing a search, noticing a wrong step — become how the model works by default.

## What you saw

- The thinking is ordinary tokens in the same stream, between `<think>` tags.
- The same weights do better with the working turned on, and the switch is the only difference.
- The working costs tokens, which cost time.
- The training signal needs nothing but a marker for the final answer.